# k-NN vs Voting V5

For V5, I wanted to see if removing some common feature values would help k-NN. I subtracted half of the training feature average after taking square roots, then normalized each image and compared k-NN alone with the same vote from V4.

First, I imported the libraries and functions I'll need.

In [1]:
from pathlib import Path
from time import perf_counter
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data_loader import load_task, split_train_val
from src.models.knn import KNNClassifier, euclidean_distance

result_dir = project_root / 'results/model_comparison'
result_dir.mkdir(parents=True, exist_ok=True)

I loaded the same image split and checked the saved V4 and NN predictions against the validation labels.

In [2]:
X, y, X_test, test_ids = load_task('image')
X_train, X_val, y_train, y_val = split_train_val(X, y, val_fraction=0.2, seed=42)
assert X_train.shape == (16000, 500) and X_val.shape == (4000, 500)
assert X.shape == (20000, 500) and X_test.shape == (5000, 500)
assert X.dtype == X_train.dtype == X_val.dtype == X_test.dtype == np.float32
v4_knn = pd.read_csv(result_dir / 'knn_vs_voting_v4_knn_validation.csv')
v4_vote = pd.read_csv(result_dir / 'knn_vs_voting_v4_image_validation.csv')
nn3 = pd.read_csv(project_root / 'results/image/nn_v3_validation.csv')
nn4 = pd.read_csv(project_root / 'results/image/nn_v4_validation.csv')
for frame in [v4_knn, v4_vote, nn3, nn4]:
    assert frame.columns.tolist() == ['validation_index', 'actual_label', 'predicted_label']
    assert len(frame) == 4000 and not frame.isna().any().any()
    assert frame.validation_index.is_unique
    assert np.array_equal(frame.validation_index, np.arange(4000))
    assert np.array_equal(frame.actual_label, y_val)
    for column in frame.columns:
        assert pd.api.types.is_integer_dtype(frame[column])
    assert frame.actual_label.between(0, 49).all() and frame.predicted_label.between(0, 49).all()

I calculated the feature average from the square-rooted training images, subtracted half of it from both sets, and normalized each row.

In [ ]:
def normalize_rows(features):
    lengths = np.linalg.norm(features, axis=1, keepdims=True)
    assert np.isfinite(features).all() and np.isfinite(lengths).all() and (lengths > 0).all()
    normalized = features / lengths
    assert normalized.shape == features.shape and normalized.dtype == np.float32
    assert np.allclose(np.linalg.norm(normalized, axis=1), 1, atol=1e-6)
    return normalized

def nearest_images(model, features):
    nearest_distances = []
    nearest_labels = []
    for image in features:
        distances = euclidean_distance(image, model.X_train)
        nearest = int(np.argmin(distances))
        nearest_distances.append(distances[nearest])
        nearest_labels.append(model.y_train[nearest])
    return np.asarray(nearest_distances), np.asarray(nearest_labels)

assert (X_train >= 0).all() and (X_val >= 0).all()
sqrt_train = np.sqrt(X_train)
feature_mean = sqrt_train.mean(axis=0)
assert feature_mean.shape == (500,) and feature_mean.dtype == np.float32
train_centered = sqrt_train - 0.5 * feature_mean
val_centered = np.sqrt(X_val) - 0.5 * feature_mean
train_centered = normalize_rows(train_centered)
val_centered = normalize_rows(val_centered)
model = KNNClassifier(k=1, distance_fn=euclidean_distance, voting_method='majority')
model.fit(train_centered, y_train)
start = perf_counter()
knn_predictions = np.asarray(model.predict(val_centered))
validation_distances, nearest_labels = nearest_images(model, val_centered)
assert np.array_equal(knn_predictions, nearest_labels)
assert len(validation_distances) == len(knn_predictions) == 4000
assert np.isfinite(validation_distances).all() and validation_distances.dtype == np.float32
assert pd.api.types.is_integer_dtype(knn_predictions)
assert ((knn_predictions >= 0) & (knn_predictions <= 49)).all()
knn_validation = pd.DataFrame({'validation_index': np.arange(4000),
    'actual_label': y_val, 'predicted_label': knn_predictions})
knn_validation.to_csv(result_dir / 'knn_vs_voting_v5_knn_validation.csv', index=False)
cutoff = np.percentile(validation_distances[:2000], 85)
print('Centered validation distance cutoff:', cutoff)
print(f'Validation neighbors calculated in {perf_counter() - start:.6f} s')

Then I applied the same agreement rule using the cutoff from the new centered distances.

In [ ]:
def conditional_vote(knn_labels, distances, nn3_labels, nn4_labels, cutoff):
    assert len(knn_labels) == len(distances) == len(nn3_labels) == len(nn4_labels)
    use_nn = (distances >= cutoff) & (nn3_labels == nn4_labels)
    predictions = knn_labels.copy()
    predictions[use_nn] = nn3_labels[use_nn]
    return predictions, use_nn

v5_predictions, use_nn = conditional_vote(
    knn_predictions, validation_distances, nn3.predicted_label.to_numpy(),
    nn4.predicted_label.to_numpy(), cutoff)
assert np.array_equal(v5_predictions[~use_nn], knn_predictions[~use_nn])
assert np.array_equal(v5_predictions[use_nn], nn3.predicted_label.to_numpy()[use_nn])
assert ((v5_predictions >= 0) & (v5_predictions <= 49)).all()
validation = pd.DataFrame({'validation_index': np.arange(4000),
    'actual_label': y_val, 'predicted_label': v5_predictions})
validation.to_csv(result_dir / 'knn_vs_voting_v5_image_validation.csv', index=False)
print('Image labels changed by the conditional vote:', int((v5_predictions != knn_predictions).sum()))
print('Validation image labels changed from V4:', int((v5_predictions != v4_vote.predicted_label.to_numpy()).sum()))

I compared the new predictions with V4 and checked both halves of the same validation set.

In [ ]:
rows = []
for name, predictions, setting in [
    ('V4 square-root k-NN', v4_knn.predicted_label.to_numpy(), 'Square root then row L2 normalization; k=1; 16000 training images'),
    ('V4 conditional vote', v4_vote.predicted_label.to_numpy(), 'V4 features with NN V3/V4 agreement at the distance cutoff'),
    ('V5 half-centered k-NN', knn_predictions, 'Square root, subtract half the training feature mean, then row L2 normalization; k=1'),
    ('V5 conditional vote', v5_predictions, 'V5 features with NN V3/V4 agreement at or above the 85th percentile of the first 2000 validation distances')]:
    correct = int((predictions == y_val).sum())
    rows.append({'candidate': name, 'correct_count': correct, 'validation_samples': len(y_val),
                 'validation_accuracy': correct / len(y_val), 'setting': setting})
summary = pd.DataFrame(rows)
summary.to_csv(result_dir / 'knn_vs_voting_v5_summary.csv', index=False)
print(summary[['candidate', 'correct_count', 'validation_samples', 'validation_accuracy']].to_string(index=False))
for start, end in [(0, 2000), (2000, 4000)]:
    old_correct = int((v4_vote.predicted_label.to_numpy()[start:end] == y_val[start:end]).sum())
    new_correct = int((v5_predictions[start:end] == y_val[start:end]).sum())
    print(f'Validation positions {start}..{end-1}: V4={old_correct}, V5={new_correct}, gain={new_correct-old_correct}')

for name, old_predictions, new_predictions in [
    ('k-NN', v4_knn.predicted_label.to_numpy(), knn_predictions),
    ('Conditional vote', v4_vote.predicted_label.to_numpy(), v5_predictions)]:
    old_correct = old_predictions == y_val
    new_correct = new_predictions == y_val
    print(f'{name}: {int((old_predictions != new_predictions).sum())} changed predictions; '
          f'{int((~old_correct & new_correct).sum())} newly correct; '
          f'{int((old_correct & ~new_correct).sum())} newly incorrect; '
          f'net gain={int(new_correct.sum() - old_correct.sum())}')


I plotted the four accuracies to compare partial mean-centering with V4.

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(['V4 k-NN', 'V4 conditional', 'V5 k-NN', 'V5 conditional'], summary.validation_accuracy)
plt.xlabel('Image approach')
plt.ylabel('Validation accuracy')
plt.title('k-NN vs Voting V5')
plt.ylim(0, 0.65)
plt.tight_layout()
plt.savefig(result_dir / 'knn_vs_voting_v5_accuracy.png')
plt.show()

I kept NN V3 for sentiment and calculated the equal average of the two task accuracies.

In [ ]:
sentiment = pd.read_csv(project_root / 'results/sentiment/nn_v3_validation.csv')
raw_sentiment_labels = pd.read_csv(project_root / 'data/raw/sentiment_train.csv', usecols=['label']).label.to_numpy()
_, _, _, sentiment_y_val = split_train_val(
    np.arange(len(raw_sentiment_labels)).reshape(-1, 1), raw_sentiment_labels,
    val_fraction=0.2, seed=42)
assert len(sentiment) == 2400 and not sentiment.isna().any().any()
assert sentiment.validation_index.is_unique
assert np.array_equal(sentiment.validation_index, np.arange(2400))
assert np.array_equal(sentiment.actual_label, sentiment_y_val)
for column in ['validation_index', 'actual_label', 'predicted_label']:
    assert pd.api.types.is_integer_dtype(sentiment[column])
assert sentiment.predicted_label.between(0, 2).all()
sentiment_correct = int((sentiment.predicted_label == sentiment.actual_label).sum())
sentiment_accuracy = sentiment_correct / len(sentiment)
image_accuracy = float((v5_predictions == y_val).mean())
combined_accuracy = (image_accuracy + sentiment_accuracy) / 2
print(f'NN V3 sentiment: {sentiment_correct}/{len(sentiment)}, accuracy={sentiment_accuracy!r}')
print('Combined validation accuracy:', combined_accuracy)

For the test images, I recalculated the feature average from all 20000 training images and used it for the same transformation.

In [ ]:
X, y, X_test, test_ids = load_task('image')
assert X.shape == (20000, 500) and X_test.shape == (5000, 500)
assert X.dtype == X_test.dtype == np.float32
assert (X >= 0).all() and (X_test >= 0).all()
sqrt_all = np.sqrt(X)
full_feature_mean = sqrt_all.mean(axis=0)
assert full_feature_mean.shape == (500,) and full_feature_mean.dtype == np.float32
all_centered = normalize_rows(sqrt_all - 0.5 * full_feature_mean)
test_centered = normalize_rows(np.sqrt(X_test) - 0.5 * full_feature_mean)
test_model = KNNClassifier(k=1, distance_fn=euclidean_distance, voting_method='majority')
test_model.fit(all_centered, y)
start = perf_counter()
test_knn_labels = np.asarray(test_model.predict(test_centered))
test_distances, test_nearest_labels = nearest_images(test_model, test_centered)
assert np.array_equal(test_knn_labels, test_nearest_labels)
assert len(test_distances) == len(test_knn_labels) == len(test_ids) == 5000
assert np.isfinite(test_distances).all() and test_distances.dtype == np.float32
test_cutoff = np.percentile(test_distances, 85)
print('Centered test distance cutoff:', test_cutoff)
print(f'Test neighbors calculated in {perf_counter() - start:.6f} s')

I matched NN V3 and V4 by image ID and applied the same conditional rule.

In [ ]:
sample = pd.read_csv(project_root / 'data/raw/sample_submission.csv')
text_ids = pd.read_csv(project_root / 'data/raw/sentiment_test.csv', usecols=['id']).id
assert sample.columns.tolist() == ['id', 'label']
assert len(sample) == 10000 and sample.id.is_unique and not sample.isna().any().any()
assert len(set(test_ids)) == 5000 and len(text_ids) == 5000 and text_ids.is_unique
assert all(str(sample_id).startswith('img_') for sample_id in test_ids)
assert text_ids.str.startswith('text_').all()
assert set(test_ids).isdisjoint(set(text_ids))
assert set(sample.id) == set(test_ids) | set(text_ids)
is_image = sample.id.isin(test_ids)
is_text = sample.id.isin(text_ids)

def align_submission(path, ids):
    frame = pd.read_csv(path)
    assert frame.columns.tolist() == ['id', 'label']
    assert len(frame) == 10000 and frame.id.is_unique and not frame.isna().any().any()
    assert set(frame.id) == set(sample.id)
    assert pd.api.types.is_integer_dtype(frame.label)
    assert frame.loc[frame.id.isin(test_ids), 'label'].between(0, 49).all()
    assert frame.loc[frame.id.isin(text_ids), 'label'].between(0, 2).all()
    aligned = pd.DataFrame({'id': ids}).merge(frame, on='id', how='left', sort=False, validate='one_to_one')
    assert aligned.id.tolist() == list(ids) and not aligned.isna().any().any()
    return aligned

nn3_images = align_submission(project_root / 'submission/nn/submission_v3_nn.csv', test_ids)
nn4_images = align_submission(project_root / 'submission/nn/submission_v4_nn.csv', test_ids)
test_predictions, test_use_nn = conditional_vote(test_knn_labels, test_distances,
    nn3_images.label.to_numpy(), nn4_images.label.to_numpy(), test_cutoff)
assert np.array_equal(test_predictions[~test_use_nn], test_knn_labels[~test_use_nn])
assert np.array_equal(test_predictions[test_use_nn], nn3_images.label.to_numpy()[test_use_nn])
print('Image labels changed from half-centered k-NN:', int((test_predictions != test_knn_labels).sum()))

Finally, I combined the new image predictions with NN V3 sentiment and saved the Kaggle CSV.

In [ ]:
image_predictions = pd.DataFrame({'id': test_ids, 'label': test_predictions})
sentiment_predictions = align_submission(project_root / 'submission/nn/submission_v3_nn.csv', text_ids)
predictions = pd.concat([image_predictions, sentiment_predictions], ignore_index=True)
assert len(predictions) == 10000 and predictions.id.is_unique
submission = sample[['id']].merge(predictions, on='id', how='left', sort=False, validate='one_to_one')
assert submission.columns.tolist() == ['id', 'label']
assert submission.id.tolist() == sample.id.tolist()
assert not submission.isna().any().any() and pd.api.types.is_integer_dtype(submission.label)
assert is_image.sum() == is_text.sum() == 5000
assert submission.loc[is_image, 'label'].between(0, 49).all()
assert submission.loc[is_text, 'label'].between(0, 2).all()
assert np.array_equal(submission.loc[is_text, 'label'],
    align_submission(project_root / 'submission/nn/submission_v3_nn.csv', sample.id).loc[is_text, 'label'])
previous = align_submission(project_root / 'submissions/submission_knn_vs_voting_v4.csv', sample.id)
changed = submission.label != previous.label
changed_images = int((changed & is_image).sum())
changed_sentiment = int((changed & is_text).sum())
print('Changed image labels compared with V4:', changed_images)
print('Changed sentiment labels compared with V4:', changed_sentiment)
assert changed_images > 0 and changed_sentiment == 0
submission_dir = project_root / 'submissions'
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / 'submission_knn_vs_voting_v5.csv'
if submission_path.exists():
    pd.testing.assert_frame_equal(pd.read_csv(submission_path), submission)
    print('Existing V5 submission matches the new predictions; kept the file.')
else:
    submission.to_csv(submission_path, index=False, mode='x')
assert pd.read_csv(submission_path).equals(submission)
print('Saved', len(submission), 'predictions to', submission_path.relative_to(project_root))

Partial mean-centering gave k-NN 57.100% accuracy, and the conditional vote reached 57.650%, with 16 more correct images than V4. The validation change was small, so I will compare the Kaggle result before deciding which approach to keep.